# CCNY Fall 2021 Calendar

Scrape the academic calendar using requests and Beautiful Soup, then organize the results into a pandas DataFrame.

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [4]:
url = "https://www.ccny.cuny.edu/registrar/fall"

headers = {"User-Agent": "curl/8.10.1"}

response = requests.get(url, headers=headers, timeout=30)

print(response.status_code)

200


In [5]:
soup = BeautifulSoup(response.text, "html.parser")

print(soup.title.get_text(strip=True))

tables = soup.find_all("table")
print("Number of tables:", len(tables))

Fall 2021 Academic Calendar | The City College of New York
Number of tables: 1


In [6]:
table = tables[0]
rows = table.find_all("tr")

for row in rows[:5]:
    cells = row.find_all(["th", "td"])
    print([cell.get_text(" ", strip=True) for cell in cells])

['DATES', 'DAYS', '']
['August 01', 'Sunday', 'Application for degree for January and February 2022 begins']
['August 18', 'Wednesday', 'Last day to apply for Study Abroad']
['August 24', 'Tuesday', 'Last day of Registration; Last day to file ePermit for the Fall 2021; Last day to drop classes for 100% tuition refund;']
['August 25', 'Wednesday', 'Start of Fall Term; Classes begin; Initial Registration Appeals begin;']


In [8]:
records = []

for row in rows[1:]:
    cells = row.find_all("td")

    if len(cells) == 3:
        records.append([
            cell.get_text(" ", strip=True)
            for cell in cells
        ])

raw_df = pd.DataFrame(
    records,
    columns=["date_text", "dow", "text"]
)

print("Rows:", len(raw_df))
raw_df.head()

Rows: 36


,date_text,dow,text
0,August 01,Sunday,Application for degree for January and Februar...
1,August 18,Wednesday,Last day to apply for Study Abroad
2,August 24,Tuesday,Last day of Registration; Last day to file ePe...
3,August 25,Wednesday,Start of Fall Term; Classes begin; Initial Reg...
4,August 25 - 31,Wednesday - Tuesday,Change of program period; late fees apply


In [9]:
raw_df["date_text"].tolist()

['August 01',
 'August 18',
 'August 24',
 'August 25',
 'August 25 - 31',
 'August 26',
 'August 28',
 'August 31',
 'September 01',
 'September 03 - 08',
 'September 06',
 'September 09',
 'September 14',
 'September 15',
 'September 15 - 16',
 'September 23',
 'September 24',
 'October 01',
 'October 08',
 'October 11',
 'November 01',
 'November 02',
 'November 04',
 'November 06',
 'November 23',
 'November 25 - 28',
 'December 11',
 'December 13',
 'December 14',
 'December 15 - 21',
 'December 21',
 'December 24 - 25',
 'December 27',
 'December 28',
 'December 31',
 'January 1, 2022']

In [11]:
from datetime import datetime, timedelta

def parse_date(date_text):
    if "," not in date_text:
        date_text = f"{date_text}, 2021"

    return datetime.strptime(date_text, "%B %d, %Y").date()

In [12]:
print(parse_date("August 01"))
print(parse_date("January 1, 2022"))
print(type(parse_date("August 01")))

2021-08-01
2022-01-01
<class 'datetime.date'>


In [13]:
def expand_dates(date_text):
    if " - " not in date_text:
        return [parse_date(date_text)]

    start_text, end_day = date_text.split(" - ")
    month = start_text.split()[0]

    start = parse_date(start_text)
    end = parse_date(f"{month} {end_day}")

    dates = []
    current = start

    while current <= end:
        dates.append(current)
        current += timedelta(days=1)

    return dates

In [14]:
for day in expand_dates("August 25 - 31"):
    print(day)

2021-08-25
2021-08-26
2021-08-27
2021-08-28
2021-08-29
2021-08-30
2021-08-31


In [15]:
expanded_records = []

for _, row in raw_df.iterrows():
    for day in expand_dates(row["date_text"]):
        expanded_records.append({
            "date": day,
            "dow": day.strftime("%A"),
            "text": row["text"]
        })

expanded_df = pd.DataFrame(expanded_records)
expanded_df = expanded_df.set_index("date").sort_index()

print("Rows:", len(expanded_df))
expanded_df.head(10)

Rows: 58


,dow,text
date,,
2021-08-01,Sunday,Application for degree for January and Februar...
2021-08-18,Wednesday,Last day to apply for Study Abroad
2021-08-24,Tuesday,Last day of Registration; Last day to file ePe...
2021-08-25,Wednesday,Start of Fall Term; Classes begin; Initial Reg...
2021-08-25,Wednesday,Change of program period; late fees apply
2021-08-26,Thursday,Change of program period; late fees apply
2021-08-26,Thursday,Last day for Independent Study
2021-08-27,Friday,Change of program period; late fees apply
2021-08-28,Saturday,Change of program period; late fees apply
